# IR Project 2026 - Colab Training Workflow

هذا النوتبوك للتجريب والتدريب فقط. النسخة النهائية للمقابلة يجب أن تكون Python scripts محلية كما طلبت الدكتورة.

الفكرة: نجرب تحميل البيانات، بناء الفهارس، التقييم، ثم نحفظ الملفات الناتجة artifacts حتى يقرأها المشروع النهائي بسرعة بدون تدريب أثناء العرض.

## 1. Install Dependencies

In [ ]:
!pip -q install numpy scipy scikit-learn matplotlib pandas streamlit joblib sentence-transformers

## 2. Clone or Upload Project

إذا رفعتم المشروع على GitHub، استخدمي clone. إذا لا، ارفعي مجلد المشروع كـ zip إلى Colab وفكي الضغط.

In [ ]:
# Option A: GitHub
# !git clone YOUR_REPO_URL ir_project
# %cd ir_project

# Option B: Upload zip manually from the left Files panel, then unzip:
# !unzip -q ir_project.zip -d /content/
# %cd /content/ir_project

import os
os.getcwd()

## 3. Download MS MARCO Files

الملف الكبير حجمه حوالي 1GB، وقد يأخذ وقتاً. للتجريب السريع يمكن تحميل qrels وqueries فقط، لكن لبناء الفهرس نحتاج collection.tsv.

In [ ]:
!python scripts/download_msmarco.py

## 4. Build Indexes

للتجريب السريع استخدمي 50000. للنسخة النهائية استخدمنا 250000 وثيقة، وهذا يتجاوز شرط 200K.

In [ ]:
# Quick training run
!PYTHONPATH=src python scripts/prepare.py --local-msmarco --max-docs 50000 --max-queries 20 --embedding-dims 64

In [ ]:
# Final-like run, takes longer
# !PYTHONPATH=src python scripts/prepare.py --local-msmarco --max-docs 250000 --max-queries 43 --embedding-dims 128

## 5. Evaluate Models

هنا نحسب MAP و nDCG و Precision@10 و Recall لكل طريقة بحث.

In [ ]:
!PYTHONPATH=src python scripts/evaluate.py --local-msmarco --max-queries 20

## 6. Optional BERT Reranking

In [ ]:
!PYTHONPATH=src python scripts/download_bert_model.py
!PYTHONPATH=src python scripts/search.py "what is diabetes treatment" --method bert_rerank --top-k 5

## 7. Test Search

In [ ]:
!PYTHONPATH=src python scripts/search.py "what are symptoms of diabetes" --method bm25 --top-k 5

## 8. Save Artifacts to Google Drive

بعد التدريب نحفظ artifacts حتى نستخدمها في النسخة النهائية بدون إعادة بناء.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ir_project_artifacts
!cp -r artifacts reports /content/drive/MyDrive/ir_project_artifacts/

## What To Say In The Interview

- استخدمنا Colab فقط للتجريب وبناء الفهارس.
- بعد انتهاء التدريب حفظنا artifacts مثل search_index و SQLite وملفات التقييم.
- النسخة النهائية تحولت إلى Python scripts منظمة وفق SOA.
- وقت المقابلة لا يتم أي تدريب online، فقط قراءة الملفات الجاهزة والبحث بسرعة.
- اخترنا MS MARCO لأنها تحتوي أكثر من 200K documents وفيها qrels للتقييم.